# Transfer-Performance Matching
This notebook focuses on creating performance block/age/season/transfer value pairs,
which can ultimately be used to train a supervised Bayesian linear regression
model to predict transfer values.

In [1]:
import pandas as pd
import duckdb
from understatapi import UnderstatClient

In [2]:
# 1. Get forward data
positions = ['F']
f_stats_df = pd.read_csv(f"../data/{'_'.join(positions)}_stats.csv")

f_stats_df.head()

,player_id,date,goals_per_90,xG_per_90,assists_per_90,xA_per_90,key_passes_per_90,xGChain_per_90,xGBuildup_per_90
0,3769,2017-11-05,0.176125,0.186441,0.35225,0.123798,2.113503,0.423613,0.130932
1,3769,2018-02-18,0.200445,0.136871,0.00000,0.056769,1.603563,0.334553,0.154694
2,3769,2019-08-25,0.000000,0.264276,0.00000,0.059867,1.022727,0.437595,0.128894
3,3769,2019-12-14,0.000000,0.061819,0.00000,0.191598,1.152503,0.591171,0.429220
4,3769,2020-07-04,0.000000,0.038386,0.00000,0.066023,0.978261,0.487978,0.383570


In [3]:
# Have to get name info, not just id, to join
id_names = pd.read_csv("../data/id_names.csv")
id_names["player_id"] = id_names["player_id"].astype("int64")

f_stats_names = pd.merge(f_stats_df, id_names, on="player_id")

f_stats_names = f_stats_names.sort_values(by="date")
f_stats_names["date"] = f_stats_names['date'].astype('datetime64[us]')
f_stats_names.head()

,player_id,date,goals_per_90,xG_per_90,assists_per_90,xA_per_90,key_passes_per_90,xGChain_per_90,xGBuildup_per_90,Unnamed: 0,name
8092,4770,2014-10-17,0.242588,0.376075,0.121294,0.139135,0.606469,0.489586,0.066570,8226,Yoann Touzghar
5930,3293,2014-10-17,0.393586,0.268840,0.000000,0.130752,1.574344,0.646234,0.312876,789,Lucas Moura
10882,3294,2014-10-17,0.430622,0.511214,0.000000,0.047331,0.645933,0.525008,0.134515,1167,Edinson Cavani
11519,3309,2014-10-18,0.234375,0.143170,0.234375,0.033735,0.468750,0.292463,0.164171,7230,Valère Germain
2633,2272,2014-10-18,0.131195,0.285712,0.131195,0.147315,0.655977,0.453249,0.169004,2107,Yannick Carrasco


In [ ]:
con = duckdb.connect()

q = """
SELECT *
FROM read_csv_auto('https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data/players.csv.gz') P
JOIN read_csv_auto('https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data/player_valuations.csv.gz') V ON P.player_id = V.player_id
LIMIT 5
"""
transfer_df = con.sql(q).df()
transfer_df.head()

,player_id,first_name,last_name,name,last_season,current_club_id,player_code,country_of_birth,city_of_birth,country_of_citizenship,...,current_club_domestic_competition_id,current_club_name,market_value_in_eur,highest_market_value_in_eur,player_id_1,date,market_value_in_eur_1,current_club_name_1,current_club_id_1,player_club_domestic_competition_id
0,405973,Fadel,Gobitaka,Fadel Gobitaka,2017,3057,fadel-gobitaka,Belgium,Sint-Agatha-Berchem,Togo,...,BE1,Royal Standard Club de Liège,50000,250000,405973,2000-01-20,150000,Unknown,3057,BE1
1,342216,Julien,Serrano,Julien Serrano,2020,1241,julien-serrano,France,Puyricard,France,...,SC1,Livingston Football Club,250000,2000000,342216,2001-07-20,100000,Unknown,1241,SC1
2,3132,Florin,Cernat,Florin Cernat,2013,126,florin-cernat,Romania,Galaţi,Romania,...,TR1,Çaykur Rizespor Kulübü,100000,2800000,3132,2003-12-09,400000,Dynamo Kyiv,126,TR1
3,6893,Gabriel,Tamas,Gabriel Tamas,2012,984,gabriel-tamas,Romania,Brașov,Romania,...,GB1,West Bromwich Albion,100000,4000000,6893,2003-12-15,900000,Galatasaray,984,GB1
4,10,Miroslav,Klose,Miroslav Klose,2015,398,miroslav-klose,Poland,Opole,Germany,...,IT1,Società Sportiva Lazio S.p.A.,1000000,30000000,10,2004-10-04,7000000,SV Werder Bremen,398,IT1


In [5]:
transfer_df.columns

Index(['player_id', 'first_name', 'last_name', 'name', 'last_season',
       'current_club_id', 'player_code', 'country_of_birth', 'city_of_birth',
       'country_of_citizenship', 'date_of_birth', 'sub_position', 'position',
       'foot', 'height_in_cm', 'contract_expiration_date', 'agent_name',
       'image_url', 'international_caps', 'international_goals',
       'current_national_team_id', 'url',
       'current_club_domestic_competition_id', 'current_club_name',
       'market_value_in_eur', 'highest_market_value_in_eur', 'player_id_1',
       'date', 'market_value_in_eur_1', 'current_club_name_1',
       'current_club_id_1', 'player_club_domestic_competition_id'],
      dtype='object')

In [ ]:
# 2. Pull forward transfer value data
con = duckdb.connect()

# We want ALL transfer data, regardless of league, for ANY player who played
# in top 5 leagues in 2021 or later
# so first need to get IDs and then filter off those

q = """
SELECT name, date_of_birth, V.date, V.market_value_in_eur AS value, V.player_club_domestic_competition_id league
FROM read_csv_auto('https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data/players.csv.gz') P
JOIN read_csv_auto('https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data/player_valuations.csv.gz') V ON P.player_id = V.player_id
WHERE P.position = 'Attack'
AND P.player_id IN (SELECT DISTINCT player_id FROM read_csv_auto('https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data/player_valuations.csv.gz') WHERE player_club_domestic_competition_id IN ('GB1', 'ES1', 'GR1', 'FR1', 'IT1') AND YEAR(V.date) >= 2021)
ORDER BY V.date ASC
"""

transfer_df = con.sql(q).df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,name,date_of_birth,date,value,league
0,Henry Giménez,1986-03-13,2021-01-01,50000,IT1
1,Jacques Zoua,1991-09-06,2021-01-01,250000,FR1
2,Xisco,1986-06-26,2021-01-02,75000,ES1
3,Leandro Rodríguez,1992-11-19,2021-01-02,275000,GB1
4,Brian Ocampo,1999-06-25,2021-01-02,400000,None


In [ ]:
# Already ordered by date
transfer_df["date"] = transfer_df['date'].astype('datetime64[us]')
transfer_df["date_of_birth"] = transfer_df['date_of_birth'].astype('datetime64[us]')
transfer_df.head()

In [ ]:
# Add age and year (fractional columns
transfer_df["age"] = (transfer_df["date"] - transfer_df["date_of_birth"]).dt.days  / 365.25 # .25 accounts for leap years

transfer_df["year"] = transfer_df["date"].dt.year + (transfer_df["date"].dt.day_of_year / 365.25)

transfer_df.head()

,name,date_of_birth,date,value,league,age,year
0,Henry Giménez,1986-03-13,2021-01-01,50000,IT1,34.806297,2021.002738
1,Jacques Zoua,1991-09-06,2021-01-01,250000,FR1,29.322382,2021.002738
2,Xisco,1986-06-26,2021-01-02,75000,ES1,34.521561,2021.005476
3,Leandro Rodríguez,1992-11-19,2021-01-02,275000,GB1,28.120465,2021.005476
4,Brian Ocampo,1999-06-25,2021-01-02,400000,None,21.524983,2021.005476


In [27]:
# 3. Join transfer data with performance data - join on latest performance date that's no later than the value date
df_combined = pd.merge_asof(f_stats_names, transfer_df, on="date", by="name")

In [28]:
# Drop those with nas
df_combined = df_combined.dropna()

In [29]:
# Rearrange columns
df_combined = df_combined[["player_id","name","date","year","age","league", "goals_per_90","xG_per_90","assists_per_90","xA_per_90","key_passes_per_90","xGChain_per_90","xGBuildup_per_90","value"]]
df_combined.head()

,player_id,name,date,year,age,league,goals_per_90,xG_per_90,assists_per_90,xA_per_90,key_passes_per_90,xGChain_per_90,xGBuildup_per_90,value
6429,3226,Ousmane Dembélé,2021-01-06,2021.013689,23.644079,ES1,0.327869,0.325879,0.000000,0.184208,2.131148,0.807533,0.413355,50000000.0
6432,7208,Samuel Chukwueze,2021-01-08,2021.013689,21.626283,ES1,0.282132,0.153102,0.282132,0.081082,1.128527,0.276931,0.042747,20000000.0
6455,5212,Sergi Guardiola,2021-01-10,2021.013689,29.607118,ES1,0.000000,0.224506,0.159574,0.107983,0.957447,0.359904,0.048266,4000000.0
6461,2589,Rafa Mir,2021-01-11,2021.013689,23.550992,ES1,0.162162,0.364977,0.000000,0.012885,0.324324,0.446071,0.101518,5500000.0
6464,2311,Roberto Soldado,2021-01-12,2021.013689,35.611225,ES1,0.344168,0.333816,0.000000,0.076386,0.516252,0.225788,0.047312,1200000.0


In [30]:
# Save to csv
df_combined.to_csv("../data/F_stats_values.csv", index=False)